Подключение библиотек

In [8]:
from __future__ import annotations

from collections import defaultdict
import numpy as np
from tqdm import tqdm
import gymnasium as gym

Создание агента и обучение


In [9]:
class BlackjackAgent:
    def __init__(
        self,
        env,
        learning_rate: float,
        initial_epsilon: float,
        epsilon_decay: float,
        final_epsilon: float,
        discount_factor: float = 0.95,
    ):
        self.q_values = defaultdict(lambda: np.zeros(env.action_space.n))
        self.lr = learning_rate
        self.discount_factor = discount_factor
        self.epsilon = initial_epsilon
        self.epsilon_decay = epsilon_decay
        self.final_epsilon = final_epsilon
        self.training_error = []

    def get_action(self, env, obs: tuple[int, int, bool]) -> int:
        if np.random.random() < self.epsilon:
            return env.action_space.sample()
        else:
            return int(np.argmax(self.q_values[obs]))

    def update(
        self,
        obs: tuple[int, int, bool],
        action: int,
        reward: float,
        terminated: bool,
        next_obs: tuple[int, int, bool],
    ):
        future_q_value = (not terminated) * np.max(self.q_values[next_obs])
        temporal_difference = (
            reward + self.discount_factor * future_q_value - self.q_values[obs][action]
        )
        self.q_values[obs][action] += self.lr * temporal_difference
        self.training_error.append(temporal_difference)

    def decay_epsilon(self):
        self.epsilon = max(self.final_epsilon, self.epsilon - self.epsilon_decay)

In [10]:
learning_rate = 0.001
n_episodes = 1000000
start_epsilon = 1.0
epsilon_decay = start_epsilon / (n_episodes / 2)
final_epsilon = 0.1

env_training = gym.make("Blackjack-v1", sab=True, render_mode='rgb_array')


In [11]:
agent = BlackjackAgent(
    env=env_training,
    learning_rate=learning_rate,
    initial_epsilon=start_epsilon,
    epsilon_decay=epsilon_decay,
    final_epsilon=final_epsilon,
)

In [12]:
env_training = gym.wrappers.RecordEpisodeStatistics(
    env_training, buffer_length=n_episodes
)

print(f"Training agent for {n_episodes} episodes...")

for episode in tqdm(range(n_episodes)):
    obs, info = env_training.reset()
    done = False

    while not done:
        action = agent.get_action(env_training, obs)
        next_obs, reward, terminated, truncated, info = env_training.step(action)
        agent.update(obs, action, reward, terminated, next_obs)
        done = terminated or truncated
        obs = next_obs

    agent.decay_epsilon()

env_training.close()
print("Agent training complete.")

Training agent for 1000000 episodes...


100%|██████████| 1000000/1000000 [02:47<00:00, 5979.62it/s]

Agent training complete.


Задание 1

In [34]:
print("\n--- Evaluating natural=True, sab=False environment ---")

env_natural = gym.make('Blackjack-v1', natural=True, sab=False)

num_games = 10
wins_natural = 0

agent.epsilon = 0

for game in range(1, num_games + 1):
    obs, info = env_natural.reset()
    done = False

    print(f"\nGame {game}")
    print(f"Start state: Player sum={obs[0]}, Dealer card={obs[1]}, Usable ace={obs[2]}")

    while not done:
        action = agent.get_action(env_natural, obs)
        action_name = "HIT" if action == 1 else "STICK"

        print(f"State: {obs} -> Action: {action_name}")

        next_obs, reward, terminated, truncated, info = env_natural.step(action)
        done = terminated or truncated
        obs = next_obs

    if reward == 1.0:
        result = "WIN"
        wins_natural += 1
    elif reward == -1.0:
        result = "LOSE"
    else:
        result = "DRAW"

    print(f"Final reward: {reward} -> {result}")

print(f"\nOut of {num_games} games (natural=True), wins: {wins_natural}")
env_natural.close()


--- Evaluating natural=True, sab=False environment ---

Game 1
Start state: Player sum=15, Dealer card=1, Usable ace=0
State: (15, 1, 0) -> Action: HIT
Final reward: -1.0 -> LOSE

Game 2
Start state: Player sum=15, Dealer card=4, Usable ace=1
State: (15, 4, 1) -> Action: HIT
State: (18, 4, 1) -> Action: STICK
Final reward: 1.0 -> WIN

Game 3
Start state: Player sum=13, Dealer card=7, Usable ace=0
State: (13, 7, 0) -> Action: HIT
State: (21, 7, 0) -> Action: STICK
Final reward: 1.0 -> WIN

Game 4
Start state: Player sum=19, Dealer card=5, Usable ace=0
State: (19, 5, 0) -> Action: STICK
Final reward: 1.0 -> WIN

Game 5
Start state: Player sum=9, Dealer card=5, Usable ace=0
State: (9, 5, 0) -> Action: HIT
State: (20, 5, 1) -> Action: STICK
Final reward: 1.0 -> WIN

Game 6
Start state: Player sum=20, Dealer card=6, Usable ace=0
State: (20, 6, 0) -> Action: STICK
Final reward: 1.0 -> WIN

Game 7
Start state: Player sum=12, Dealer card=6, Usable ace=0
State: (12, 6, 0) -> Action: STICK
Fina

Задание 2

In [31]:
print("\n--- Evaluating natural=False, sab=False environment ---")

env_no_natural = gym.make('Blackjack-v1', natural=False, sab=False)

num_games = 10
wins_no_natural = 0

agent.epsilon = 0

for game in range(1, num_games + 1):
    obs, info = env_no_natural.reset()
    done = False

    print(f"\nGame {game}")
    print(f"Start state: Player sum={obs[0]}, Dealer card={obs[1]}, Usable ace={obs[2]}")

    while not done:
        action = agent.get_action(env_no_natural, obs)
        action_name = "HIT" if action == 1 else "STICK"

        print(f"State: {obs} -> Action: {action_name}")

        next_obs, reward, terminated, truncated, info = env_no_natural.step(action)
        done = terminated or truncated
        obs = next_obs

    if reward == 1.0:
        result = "WIN"
        wins_no_natural += 1
    elif reward == -1.0:
        result = "LOSE"
    else:
        result = "DRAW"

    print(f"Final reward: {reward} -> {result}")

print(f"\nOut of {num_games} games (natural=False), wins: {wins_no_natural}")
env_no_natural.close()


--- Evaluating natural=False, sab=False environment ---

Game 1
Start state: Player sum=8, Dealer card=3, Usable ace=0
State: (8, 3, 0) -> Action: HIT
State: (13, 3, 0) -> Action: STICK
Final reward: 1.0 -> WIN

Game 2
Start state: Player sum=19, Dealer card=1, Usable ace=1
State: (19, 1, 1) -> Action: HIT
State: (16, 1, 0) -> Action: HIT
Final reward: -1.0 -> LOSE

Game 3
Start state: Player sum=12, Dealer card=10, Usable ace=0
State: (12, 10, 0) -> Action: HIT
State: (13, 10, 0) -> Action: HIT
State: (19, 10, 0) -> Action: STICK
Final reward: 1.0 -> WIN

Game 4
Start state: Player sum=20, Dealer card=9, Usable ace=0
State: (20, 9, 0) -> Action: STICK
Final reward: 1.0 -> WIN

Game 5
Start state: Player sum=18, Dealer card=10, Usable ace=0
State: (18, 10, 0) -> Action: STICK
Final reward: -1.0 -> LOSE

Game 6
Start state: Player sum=19, Dealer card=1, Usable ace=0
State: (19, 1, 0) -> Action: STICK
Final reward: 1.0 -> WIN

Game 7
Start state: Player sum=17, Dealer card=7, Usable ace